In [24]:
# ==========================================
# CELL 1: Setup & Load Data
# ==========================================
import json
import glob
import os
import pandas as pd
import numpy as np

RAW_DIR = "../data"

files = sorted(glob.glob(os.path.join(RAW_DIR, "*.jsonl")))
print(f"Found {len(files)} raw movie files.")

rows = []
for f in files:
    with open(f, "r", encoding="utf-8") as file:
        rows.extend([json.loads(line) for line in file if line.strip()])

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} movies with {df.shape[1]} columns.")

Found 1 raw movie files.
Loaded 10639 movies with 27 columns.


In [25]:
# TMDB uses 0 for missing budgets, not just free films. 
# We must replace 0 with NaN so Pandas knows they are missing, otherwise 
# they will ruin our ROI calculations (division by zero).

budget_zeros_before = (df["budget"] == 0).sum()
revenue_zeros_before = (df["revenue"] == 0).sum()
print(f"Zeros before fix -> Budget: {budget_zeros_before}, Revenue: {revenue_zeros_before}")

df["budget"] = df["budget"].replace(0, np.nan)
df["revenue"] = df["revenue"].replace(0, np.nan)

print(f"Missing after fix  -> Budget: {df['budget'].isna().sum()}, Revenue: {df['revenue'].isna().sum()}")

# How many movies are completely unusable for financial analysis?
both_missing = df["budget"].isna() & df["revenue"].isna()
print(f"Movies with BOTH missing: {both_missing.sum()}")

Zeros before fix -> Budget: 3075, Revenue: 1483
Missing after fix  -> Budget: 3075, Revenue: 1483
Movies with BOTH missing: 1260


Filtering & Extracting Nested Fields

In [26]:
# Filter to movies with at least 100 votes to ensure reliable data
n_zero = (df["vote_count"] == 0).sum()
df = df[df["vote_count"] >= 100].copy()
print(f"After vote filter (>=100), new shape: {df.shape}")

# Extract primary genre (from a list of dicts)
def primary_genre(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]["name"]
    return None
df["primary_genre"] = df["genres"].apply(primary_genre)

# Extract collection ID (to identify franchises)
def collection_id(x):
    if isinstance(x, dict):
        return x.get("id")
    return None
df["collection_id"] = df["belongs_to_collection"].apply(collection_id)
df["is_franchise"] = df["collection_id"].notna()

# Extract release year
df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.astype("Int64")

print(f"Franchises identified: {df['is_franchise'].sum()} ({df['is_franchise'].mean()*100:.1f}%)")
print(f"Release year range: {df['release_year'].min()} – {df['release_year'].max()}")

After vote filter (>=100), new shape: (9517, 27)
Franchises identified: 2662 (28.0%)
Release year range: 1980 – 2025


Calculate Financials & Outliers

In [27]:
# Calculate ROI and Profit
valid_financials = df["budget"].notna() & df["revenue"].notna()
df["roi"] = np.where(valid_financials & (df["budget"] > 0), df["revenue"] / df["budget"], np.nan)
df["profit"] = np.where(valid_financials, df["revenue"] - df["budget"], np.nan)
df["log_roi"] = np.where((df["roi"] > 0), np.log(df["roi"]), np.nan)

# Flag Suspect Outliers
# We identified data artifacts: placeholder budgets (<$10k) and streaming revenue fragments (ROI < 0.02)
df["suspect_roi"] = ((df["budget"] < 10_000) | (df["roi"] < 0.02))
print(f"Suspect ROI flagged: {df['suspect_roi'].sum()} movies ({df['suspect_roi'].mean()*100:.2f}%)")

Suspect ROI flagged: 52 movies (0.55%)


Final Filter & Save Base Clean Data

In [28]:
clean_df = df[
    df["budget"].notna() &
    df["revenue"].notna() &
    df["release_year"].notna() &
    (~df["suspect_roi"]) &
    (df["vote_count"] >= 100)
].copy()

print(f"Final clean dataset: {len(clean_df)} movies")

# Drop the messy JSON columns so the CSV is readable
cols_to_drop = ["genres", "belongs_to_collection", "production_companies", "production_countries", "spoken_languages"]
clean_df = clean_df.drop(columns=[c for c in cols_to_drop if c in clean_df.columns])

clean_df.to_csv("../data/movies_clean.csv", index=False)
print("Saved clean base dataset to ../data/movies_clean.csv")

Final clean dataset: 6990 movies
Saved clean base dataset to ../data/movies_clean.csv


Feature Engineering - Adaptations & Remakes

In [29]:
with open("../data/keywords/movie_keywords.json", "r") as f:
    keywords_dict = json.load(f)
    print(f"Loaded keywords for {len(keywords_dict)} movies")

ADAPTATION_KEYWORDS = [    
    "based on young adult novel", "based on cartoon", "based on song, poem or rhyme",
    "based on play or musical", "based on graphic novel", "based on comic",
    "based on light novel", "based on video game", "based on documentary",
    "based on manga", "based on movie", "based on fairy tale",
    "based on children's book", "based on anime", "based on short film",
    "based on short story", "based on webcomic or webtoon", "based on novel or book",
    "based on myths, legends or folklore", "based on game", "based on web series",
    "based on tv series", "based on memoir or autobiography"
]
REMAKE_KEYWORDS = [
    "remake", "live action remake", "reboot", "reimagining", 
    "reimagined", "rebooted", "remade", "remakes", "remaking", 
]

def check_derivative(kw_list):
    if not isinstance(kw_list, list):
        return False, False

    kw_names = []
    for k in kw_list:
        if isinstance(k, dict) and "name" in k:
            kw_names.append(k["name"].lower())
        elif isinstance(k, str):
            kw_names.append(k.lower())
            
    is_adaptation = any(kw in kw_names for kw in ADAPTATION_KEYWORDS)
    is_remake = any(kw in kw_names for kw in REMAKE_KEYWORDS)
    
    return is_adaptation, is_remake

# Map the JSON to the dataframe (using strings to match keys)
clean_df["kw_list"] = clean_df["id"].astype(str).map(keywords_dict)

# Apply the function
clean_df[["is_adaptation", "is_remake"]] = clean_df["kw_list"].apply(
    lambda x: pd.Series(check_derivative(x))
)

# Create the master flag
clean_df["is_derivative"] = clean_df["is_adaptation"] | clean_df["is_remake"] | clean_df["is_franchise"]

print(f"Franchises (Sequels/Prequels): {clean_df['is_franchise'].mean()*100:.1f}%")
print(f"Adaptations:                   {clean_df['is_adaptation'].mean()*100:.1f}%")
print(f"Remakes:                       {clean_df['is_remake'].mean()*100:.1f}%")
print(f"-------------------------------------------")
print(f"TOTAL Derivative Works:        {clean_df['is_derivative'].mean()*100:.1f}%")

Loaded keywords for 7310 movies
Franchises (Sequels/Prequels): 28.3%
Adaptations:                   22.8%
Remakes:                       4.5%
-------------------------------------------
TOTAL Derivative Works:        45.8%


Save Final Enriched Dataset

In [30]:
# Drop the keyword list to keep the final CSV flat
clean_df = clean_df.drop(columns=["kw_list"])

clean_df.to_csv("../data/movies_final.csv", index=False)
print(f"Successfully saved movies_final.csv with {len(clean_df)} rows and {clean_df.shape[1]} columns.")

Successfully saved movies_final.csv with 6990 rows and 33 columns.
